# On the Wings of a Butterfly — Exploring Chaos
## Lesson 5

In this notebook you will explore three iconic examples of **chaotic and fractal systems**:

1. **The Logistic Map** — a simple equation that produces period doubling and chaos
2. **The Lorenz System** — the original "butterfly effect" in a 3D flow
3. **The Mandelbrot & Julia Sets** — fractals born from iteration in the complex plane

Each section builds step by step: you implement small functions, test them on simple examples, and then combine them into full visualisations.


## 0 · Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('All imports OK ✓')


---
# Part 1 — The Logistic Map

The **logistic map** is one of the simplest equations that can produce chaos:

$$X_{N+1} = r \, X_N (1 - X_N)$$

- $X_N \in [0, 1]$ represents the normalised population size at generation $N$.
- $r \in [0, 4]$ is the growth-rate parameter.

Despite its simplicity, as $r$ increases the system goes through:
- **stable fixed points** → **period-2 oscillations** → **period-4** → **period-8** → … → **chaos**

This is called a **period-doubling cascade**.


## 1.1 · One iteration of the logistic map
We start with the smallest possible building block: computing a single $X_{N+1}$ from $X_N$.


In [ ]:
def logistic_step(x, r):
    """Compute one step of the logistic map: x_{n+1} = r * x_n * (1 - x_n)."""
    return r * x * (1 - x)

print(logistic_step(0.5, r=2.0))


### Independent test 1
If $X_N = 0$ or $X_N = 1$, the next value should always be $0$ regardless of $r$.


In [ ]:
# --- Test: boundary values ---
result_zero = logistic_step(0.0, r=3.5)
result_one  = logistic_step(1.0, r=3.5)
print(f'logistic_step(0.0, 3.5) = {result_zero}')
print(f'logistic_step(1.0, 3.5) = {result_one}')
assert result_zero == 0.0
assert result_one  == 0.0
print('Test passed ✓ Boundary values behave correctly.')


## 1.2 · Iterating the logistic map
Now we iterate the map many times and record the full trajectory $X_0, X_1, \ldots, X_N$.


In [ ]:
def logistic_trajectory(x0, r, n_steps):
    """Return an array of length n_steps+1 with the full trajectory."""
    x = np.zeros(n_steps + 1)
    x[0] = x0
    for i in range(n_steps):
        x[i + 1] = logistic_step(x[i], r)
    return x

traj = logistic_trajectory(x0=0.5, r=2.5, n_steps=30)
print(f'First 6 values: {traj[:6].round(4)}')
print(f'Last value:      {traj[-1]:.6f}')


### Independent test 2
For $r = 2.5$ the fixed point is $X^* = 1 - 1/r = 0.6$. After many steps the trajectory should converge there.


In [ ]:
# --- Test: convergence to fixed point ---
x_star = 1 - 1 / 2.5
traj_test = logistic_trajectory(0.2, r=2.5, n_steps=100)
print(f'Expected fixed point: {x_star:.4f}')
print(f'Trajectory end value: {traj_test[-1]:.6f}')
assert np.isclose(traj_test[-1], x_star, atol=1e-6)
print('Test passed ✓ Trajectory converges to the fixed point.')


## 1.3 · Visualising different regimes
Let's see how the behaviour changes as $r$ increases.


In [ ]:
r_values = [1.5, 2.8, 3.3, 3.5, 3.56, 3.9]
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)

for ax, r_val in zip(axes.flat, r_values):
    traj = logistic_trajectory(0.5, r_val, 60)
    ax.plot(traj, 'o-', markersize=3, linewidth=1, color='steelblue')
    ax.set_title(f'r = {r_val}')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('$X_N$')

axes[-1, 0].set_xlabel('Generation N')
axes[-1, 1].set_xlabel('Generation N')
plt.suptitle('Logistic Map — Time Series for Different r', fontweight='bold')
plt.tight_layout()
plt.show()


## 1.4 · The cobweb diagram
A **cobweb diagram** shows the iteration graphically:
- Plot the parabola $y = r\,x(1-x)$ and the diagonal $y = x$.
- Starting from $X_0$, go **vertically** to the parabola, then **horizontally** to the diagonal. Repeat.


In [ ]:
def plot_cobweb(x0, r, n_steps=40, ax=None):
    """Draw a cobweb diagram for the logistic map."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    x_curve = np.linspace(0, 1, 300)
    y_curve = r * x_curve * (1 - x_curve)

    ax.plot(x_curve, y_curve, 'b-', linewidth=2, label=f'$f(x) = {r}x(1-x)$')
    ax.plot([0, 1], [0, 1], 'k-', linewidth=1, label='$y = x$')

    x = x0
    ax.plot([x, x], [0, logistic_step(x, r)], 'r-', linewidth=0.8)
    for _ in range(n_steps):
        x_new = logistic_step(x, r)
        ax.plot([x, x_new], [x_new, x_new], 'r-', linewidth=0.8)
        ax.plot([x_new, x_new], [x_new, logistic_step(x_new, r)], 'r-', linewidth=0.8)
        x = x_new

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_xlabel('$X_N$')
    ax.set_ylabel('$X_{N+1}$')
    ax.set_title(f'Cobweb · r = {r}')
    ax.legend(fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r_val in zip(axes, [1.5, 3.2, 3.9]):
    plot_cobweb(0.2, r_val, ax=ax)
plt.tight_layout()
plt.show()


## 1.5 · The bifurcation diagram
The **bifurcation diagram** shows, for each $r$, the long-term values that $X_N$ visits. It reveals the period-doubling route to chaos at a glance.


In [ ]:
def bifurcation_diagram(r_min=2.5, r_max=4.0, n_r=4000, n_transient=500, n_plot=200):
    """Compute (r, x) pairs for a bifurcation diagram."""
    r_values = np.linspace(r_min, r_max, n_r)
    x = 0.5 * np.ones_like(r_values)

    # discard transients
    for _ in range(n_transient):
        x = r_values * x * (1 - x)

    rs, xs = [], []
    for _ in range(n_plot):
        x = r_values * x * (1 - x)
        rs.append(r_values.copy())
        xs.append(x.copy())

    return np.concatenate(rs), np.concatenate(xs)

r_pts, x_pts = bifurcation_diagram()

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(r_pts, x_pts, ',', color='black', markersize=0.02, alpha=0.5)
ax.set_xlabel('r')
ax.set_ylabel('$X_N$ (long-term)')
ax.set_title('Bifurcation Diagram of the Logistic Map')
ax.set_xlim(2.5, 4.0)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 1.6 · Sensitivity to initial conditions
In the chaotic regime, two trajectories starting very close together diverge rapidly.


In [ ]:
r_chaos = 3.9
traj_a = logistic_trajectory(0.5000000, r_chaos, 60)
traj_b = logistic_trajectory(0.5000001, r_chaos, 60)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(traj_a, 'o-', markersize=3, label='$X_0 = 0.5000000$', color='steelblue')
axes[0].plot(traj_b, 'o-', markersize=3, label='$X_0 = 0.5000001$', color='firebrick')
axes[0].set_ylabel('$X_N$')
axes[0].set_title('Two nearby trajectories in the chaotic regime (r = 3.9)')
axes[0].legend()

axes[1].plot(np.abs(traj_a - traj_b), 'o-', markersize=3, color='purple')
axes[1].set_ylabel('$|X_A - X_B|$')
axes[1].set_xlabel('Generation N')
axes[1].set_title('Difference between the two trajectories')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print('Observation: after ~20 generations the tiny initial difference has grown to order 1.')


### What to try
- Change `r_chaos` to values like `3.5` (periodic) and compare the divergence plot.
- Try `r = 4.0` — is chaos stronger or weaker?
- Try different initial separations (e.g. $10^{-12}$) — how many extra steps do you gain?


---
# Part 2 — The Lorenz System

Edward Lorenz's simplified model of atmospheric convection:

$$\frac{dx}{dt} = \sigma(y - x)$$
$$\frac{dy}{dt} = \rho\,x - y - x\,z$$
$$\frac{dz}{dt} = x\,y - b\,z$$

Classical chaotic parameters: $\sigma = 10$, $\rho = 28$, $b = 8/3$.


## 2.1 · Define the Lorenz equations


In [ ]:
def lorenz(t, state, sigma=10.0, rho=28.0, b=8/3):
    """Right-hand side of the Lorenz system."""
    x, y, z = state
    dx = sigma * (y - x)
    dy = rho * x - y - x * z
    dz = x * y - b * z
    return [dx, dy, dz]


### Independent test 3
At the origin $(0, 0, 0)$ all derivatives should be zero (it is an equilibrium point).


In [ ]:
# --- Test: equilibrium at origin ---
result = lorenz(0, [0.0, 0.0, 0.0])
print('Derivatives at origin:', result)
assert all(v == 0.0 for v in result)
print('Test passed ✓ The origin is an equilibrium.')


## 2.2 · Solve the Lorenz system


In [ ]:
T_end = 50.0
t_eval = np.linspace(0, T_end, 20000)

sol = solve_ivp(lorenz, (0, T_end), [1.0, 1.0, 1.0],
                t_eval=t_eval, max_step=0.01, rtol=1e-9)

print(f'Solver status: {"success" if sol.success else sol.message}')
print(f'Shape of solution: {sol.y.shape}')


## 2.3 · The strange attractor in 3D


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(sol.y[0], sol.y[1], sol.y[2], linewidth=0.4, color='steelblue')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Lorenz Strange Attractor')
plt.tight_layout()
plt.show()


## 2.4 · Time series of $x(t)$
The $x$ component switches unpredictably between positive and negative values — corresponding to the two "wings" of the attractor.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sol.t, sol.y[0], linewidth=0.5, color='steelblue')
ax.set_xlabel('Time')
ax.set_ylabel('x(t)')
ax.set_title('Lorenz System — x component over time')
plt.tight_layout()
plt.show()


## 2.5 · Sensitivity to initial conditions
Start two trajectories with a tiny difference ($10^{-9}$) and watch them diverge.


In [ ]:
ic_a = [1.0, 1.0, 1.0]
ic_b = [1.0, 1.0, 1.0 + 1e-9]

sol_a = solve_ivp(lorenz, (0, T_end), ic_a, t_eval=t_eval, max_step=0.01, rtol=1e-12)
sol_b = solve_ivp(lorenz, (0, T_end), ic_b, t_eval=t_eval, max_step=0.01, rtol=1e-12)

diff = np.sqrt(np.sum((sol_a.y - sol_b.y)**2, axis=0))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(sol_a.t, sol_a.y[0], linewidth=0.5, label='Trajectory A', color='steelblue')
axes[0].plot(sol_b.t, sol_b.y[0], linewidth=0.5, label='Trajectory B', color='firebrick', alpha=0.7)
axes[0].set_ylabel('x(t)')
axes[0].set_title('Two Lorenz trajectories with initial difference $10^{-9}$')
axes[0].legend()

axes[1].plot(sol_a.t, diff, linewidth=0.8, color='purple')
axes[1].set_yscale('log')
axes[1].set_ylabel('$||A - B||$')
axes[1].set_xlabel('Time')
axes[1].set_title('Distance between the two trajectories')

plt.tight_layout()
plt.show()

print('Observation: trajectories are indistinguishable at first, then diverge exponentially.')


### What to try
- Change `rho` to 15 (below the chaotic threshold) — does the attractor still look butterfly-shaped?
- Try `rho = 100` — what happens?
- Change the initial separation from $10^{-9}$ to $10^{-3}$ — how much sooner do the trajectories diverge?


---
# Part 3 — Fractals: Mandelbrot and Julia Sets

Both arise from one iteration in the complex plane:

$$z_{n+1} = z_n^2 + c$$

- **Mandelbrot set**: fix $z_0 = 0$, vary $c$. Which values of $c$ keep the iteration bounded?
- **Julia set**: fix $c$, vary $z_0$. Which starting points stay bounded?


## 3.1 · Iterate one point
We start by checking whether a single complex number $c$ belongs to the Mandelbrot set.


In [ ]:
def mandelbrot_escape(c, max_iter=100):
    """Return the escape iteration for a single complex number c.
    If the point does not escape within max_iter, return max_iter (= inside the set).
    """
    z = 0 + 0j
    for n in range(max_iter):
        z = z * z + c
        if abs(z) > 2:
            return n
    return max_iter

print('c = 0+0j  →', mandelbrot_escape(0 + 0j))
print('c = 1+0j  →', mandelbrot_escape(1 + 0j))
print('c = -1+0j →', mandelbrot_escape(-1 + 0j))


### Independent test 4
$c = 0$ is the centre of the Mandelbrot set and should never escape. $c = 2$ escapes immediately.


In [ ]:
# --- Test: known Mandelbrot membership ---
assert mandelbrot_escape(0 + 0j) == 100   # inside
assert mandelbrot_escape(2 + 0j) < 5      # escapes fast
assert mandelbrot_escape(-1 + 0j) == 100  # inside (period-2 point)
print('Test passed ✓ Known points classified correctly.')


## 3.2 · The Mandelbrot set — full image
We compute the escape time for a grid of complex numbers and visualise the result.


In [ ]:
def compute_mandelbrot(x_range=(-2, 1), y_range=(-1.5, 1.5), width=800, height=800, max_iter=100):
    """Compute escape-time array for the Mandelbrot set (vectorised)."""
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    real, imag = np.meshgrid(x, y)
    c = real + 1j * imag
    z = np.zeros_like(c)
    escape_time = np.full(c.shape, max_iter, dtype=int)

    for i in range(max_iter):
        mask = np.abs(z) <= 2
        z[mask] = z[mask] ** 2 + c[mask]
        newly_escaped = (np.abs(z) > 2) & (escape_time == max_iter)
        escape_time[newly_escaped] = i

    return x, y, escape_time

x, y, escape = compute_mandelbrot()

fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(escape, extent=[x.min(), x.max(), y.min(), y.max()],
               origin='lower', cmap='inferno', aspect='equal')
ax.set_title('The Mandelbrot Set')
ax.set_xlabel('Re(c)')
ax.set_ylabel('Im(c)')
plt.colorbar(im, ax=ax, label='Escape iteration')
plt.tight_layout()
plt.show()


## 3.3 · Zooming into the Mandelbrot set
One of the most spectacular properties of fractals is that **zooming in reveals new detail** that resembles the whole.


In [ ]:
zoom_regions = [
    {'x_range': (-2, 1),       'y_range': (-1.5, 1.5),    'title': 'Full view'},
    {'x_range': (-0.8, -0.7),  'y_range': (0.1, 0.2),     'title': 'Zoom 1 — spiral'},
    {'x_range': (-0.748, -0.745), 'y_range': (0.1, 0.103), 'title': 'Zoom 2 — deeper'},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, region in zip(axes, zoom_regions):
    x, y, esc = compute_mandelbrot(
        x_range=region['x_range'], y_range=region['y_range'],
        width=600, height=600, max_iter=300
    )
    ax.imshow(esc, extent=[x.min(), x.max(), y.min(), y.max()],
              origin='lower', cmap='inferno', aspect='equal')
    ax.set_title(region['title'])
    ax.set_xlabel('Re(c)')
    ax.set_ylabel('Im(c)')

plt.suptitle('Self-Similarity in the Mandelbrot Set', fontweight='bold')
plt.tight_layout()
plt.show()


## 3.4 · Julia sets
For a **fixed** $c$, a Julia set shows which initial values $z_0$ stay bounded.


In [ ]:
def compute_julia(c_value, x_range=(-2, 2), y_range=(-2, 2), width=800, height=800, max_iter=200):
    """Compute escape-time array for the Julia set of a given c."""
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    real, imag = np.meshgrid(x, y)
    z = real + 1j * imag
    escape_time = np.full(z.shape, max_iter, dtype=int)

    for i in range(max_iter):
        mask = np.abs(z) <= 2
        z[mask] = z[mask] ** 2 + c_value
        newly_escaped = (np.abs(z) > 2) & (escape_time == max_iter)
        escape_time[newly_escaped] = i

    return x, y, escape_time


In [ ]:
c_values = [
    (-0.7 + 0.27015j, 'c = -0.70 + 0.27i'),
    (-0.4 + 0.6j,     'c = -0.40 + 0.60i'),
    (0.355 + 0.355j,  'c =  0.355 + 0.355i'),
    (-0.8 + 0.156j,   'c = -0.80 + 0.156i'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (c_val, title) in zip(axes.flat, c_values):
    x, y, esc = compute_julia(c_val, max_iter=200)
    ax.imshow(esc, extent=[x.min(), x.max(), y.min(), y.max()],
              origin='lower', cmap='twilight_shifted', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('Re($z_0$)')
    ax.set_ylabel('Im($z_0$)')

plt.suptitle('Julia Sets for Different Values of c', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


### What to try
- Pick a $c$ **inside** the Mandelbrot set (e.g. $c = -0.1 + 0.7i$) and one **outside** (e.g. $c = 1$). Compare the Julia sets.
- Inside → **connected** fractal. Outside → **disconnected** dust.
- Increase `max_iter` for more detail near the boundary.
- Try `cmap='hot'`, `'ocean'`, `'cubehelix'` for different aesthetics.


---
## Connecting the Three Systems

All three topics are manifestations of the same mathematical idea — **nonlinear iteration**:

| System | Type | Key feature |
|--------|------|-------------|
| Logistic map | Discrete, 1D real | Period doubling → chaos |
| Lorenz system | Continuous, 3D real | Strange attractor, butterfly effect |
| Mandelbrot / Julia | Discrete, 2D complex | Fractal boundaries |

The logistic map $X_{N+1} = r\,X_N(1-X_N)$ is actually related to $z_{n+1} = z_n^2 + c$ by a simple change of variables. The bifurcation diagram of the logistic map corresponds to a **slice through the Mandelbrot set** along the real axis.


---
## What to Try Next — Guided Experiments

### Logistic map
1. Find the value of $r$ where the first period-doubling occurs (from fixed point to period-2). Hint: it is near $r = 3$.
2. Find the period-3 window in the bifurcation diagram. At what $r$ does it start?
3. Compute the Feigenbaum ratio: measure the $r$-intervals between successive doublings and compute their ratio. Does it approach $\delta \approx 4.669$?

### Lorenz system
4. Find the other two equilibrium points of the Lorenz system (they are not at the origin). Hint: set all derivatives to zero and solve.
5. Plot all three components $x(t)$, $y(t)$, $z(t)$ together. Which component is most "regular"?
6. Try $\rho = 10$ (non-chaotic) — what shape does the attractor become?

### Fractals
7. Pick a point on the boundary of the Mandelbrot set and zoom in 1000×. Describe what you see.
8. Animate a sequence of Julia sets where $c$ moves along a circle in the complex plane.


---
## Independent Work and Mini-Projects

### Project A · Lyapunov exponent of the logistic map
Compute the **Lyapunov exponent** $\lambda$ as a function of $r$. Plot $\lambda(r)$ alongside the bifurcation diagram. Chaos corresponds to $\lambda > 0$.

### Project B · Lorenz system parameter sweep
Vary $\rho$ from 0 to 50 and classify the long-term behaviour (fixed point, periodic, chaotic) for each value. Plot a "bifurcation-like" diagram for the Lorenz system.

### Project C · Mandelbrot set and the logistic map
Show that the logistic map $X_{N+1} = rX_N(1-X_N)$ can be transformed into $z_{n+1} = z_n^2 + c$ with the substitution $z = r(1/2 - X)$ and $c = r/2 - r^2/4$. Overlay the real-axis slice of the Mandelbrot set on top of the logistic-map bifurcation diagram.

### Project D · 3D fractal rendering
Render a high-resolution Mandelbrot set with smooth colouring using the "normalised iteration count" technique. Experiment with custom colour maps.

### Project E · Double pendulum
Implement another famous chaotic system — the **double pendulum**. Simulate and visualise its motion. Show sensitivity to initial conditions.

### Final Reflection
After completing one or more projects, explain:
- Why deterministic systems can be unpredictable
- What the practical limits of prediction are for chaotic systems
- Where you see chaos and fractals in real-world phenomena
